In [1]:
import os

#### Write Dockerfile

In [2]:
%%writefile Dockerfile

# ==========================
# Stage 1: Build React frontend
# ==========================
FROM node:20-alpine AS frontend-builder

WORKDIR /app/frontend

# Install frontend dependencies
COPY frontend/package*.json ./
RUN npm install --no-audit --progress=false

# Copy source and build production assets
COPY frontend/ ./
RUN npm run build


# ==========================
# Stage 2: Build Python backend (FastAPI)
# ==========================
FROM python:3.11-slim AS backend

# Prevent Python from writing .pyc files & buffering stdout/stderr
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /app

# Copy and install backend dependencies
COPY backend/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy backend source code
COPY backend/ ./

# Copy built frontend assets into the container (FastAPI serves these)
COPY --from=frontend-builder /app/frontend/dist ./frontend_build

# Expose port (App Runner compatible)
EXPOSE 5000

# Run FastAPI with gunicorn + uvicorn workers (production)
CMD ["gunicorn", "main:app", \
     "--workers", "2", \
     "--worker-class", "uvicorn.workers.UvicornWorker", \
     "--bind", "0.0.0.0:5000", \
     "--timeout", "300", \
     "--keep-alive", "65", \
     "--access-logfile", "-"]

Writing Dockerfile


#### Build image and push to ECR

In [3]:
%%sh

image=quiz-builder-image

docker build -t ${image} .

region=${region:-us-west-2}

account=$(aws sts get-caller-identity --query Account --output text)

fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE || true

docker tag  ${image} ${fullname}

docker push ${fullname}

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 1.30kB done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/node:20-alpine
#2 DONE 0.5s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 0.5s

#4 [internal] load .dockerignore
#4 transferring context: 122B done
#4 DONE 0.0s

#5 [frontend-builder 1/6] FROM docker.io/library/node:20-alpine@sha256:fb4cd12c85ee03686f6af5362a0b0d56d50c58a04632e6c0fb8363f609372293
#5 DONE 0.0s

#6 [backend 1/6] FROM docker.io/library/python:3.11-slim@sha256:6d85378d88a19cd4d76079817532d62232be95757cb45945a99fec8e8084b9c2
#6 DONE 0.0s

#7 [internal] load build context
#7 transferring context: 11.55kB done
#7 DONE 0.0s

#8 [backend 2/6] WORKDIR /app
#8 CACHED

#9 [backend 3/6] COPY backend/requirements.txt .
#9 CACHED

#10 [backend 4/6] RUN pip install --no-cache-dir -r requirements.txt
#10 CACHED

#11 [frontend-builder 3/6] COPY fro

Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'quiz-builder-image' already exists in the registry with id '672760273172'


The push refers to repository [672760273172.dkr.ecr.us-west-2.amazonaws.com/quiz-builder-image]
c78e9cf1d820: Preparing
579f8024df52: Preparing
baf8f58a3181: Preparing
a7a31e0eab17: Preparing
c6b55b7b729c: Preparing
2360b28b4660: Preparing
248c30140986: Preparing
fa1aec823035: Preparing
6d7c150df58d: Preparing
2360b28b4660: Waiting
248c30140986: Waiting
fa1aec823035: Waiting
6d7c150df58d: Waiting
a7a31e0eab17: Layer already exists
c6b55b7b729c: Layer already exists
baf8f58a3181: Layer already exists
2360b28b4660: Layer already exists
248c30140986: Layer already exists
fa1aec823035: Layer already exists
6d7c150df58d: Layer already exists
579f8024df52: Pushed
c78e9cf1d820: Pushed
latest: digest: sha256:1e1bf8dbd3c53abcbb6a80c310b72b7fe855ac08ebe2ef0b8ef01cc8075b19fd size: 2202


#### Remove Dockerfile

In [4]:
os.remove('Dockerfile')